# Gaussian MCMC: inferring tree hyperparameters

> **Status:** placeholder — content coming soon.

In [notebook 04](04_phylo_bayesian.ipynb) the model's hyperparameters (diffusivity $\sigma^2$ and observation noise $\tau^2$) were known: we used Gaussian BFFG as a *closed-form smoother* to obtain the exact posterior over ancestral states. Once the hyperparameters become **unknown**, the closed form alone is no longer enough — we need a posterior over them too, jointly with the latent states.

This notebook will introduce the **MCMC counterpart** of BFFG: the same backward filter + forward guide now lives inside the kernel of a Metropolis–Hastings chain that targets

$$p(\theta, \text{noise} \mid \text{leaf obs}), \qquad \theta = (\sigma^2, \tau^2, \ldots).$$

This is the same conceptual setup as [notebook 07](07_sde_mcmc.ipynb) (SDE transitions), only here every BFFG step is *exact* and the chain only needs to do the work that hyperparameter inference inherently demands.

## Planned outline

1. The linear-Gaussian tree model, with $(\sigma^2, \tau^2)$ now treated as parameters
2. Marginal log-likelihood via the BFFG up-sweep: $\log p(\text{obs} \mid \theta) = \log h_{\text{root}}(x_{\text{root}})$
3. Two-block Gibbs kernel:
   - pCN on the latent noise field (preserves the standard-normal prior)
   - Random walk in log-space on the positive hyperparameters
4. Single-chain trace + diagnostics
5. Multi-chain Gelman–Rubin $\hat R$ via `joblib` (same pattern as notebook 07)

## Why this comes before SDE MCMC

All the moving pieces are easier to inspect in the Gaussian setting: BFFG is closed form (no ODE / Euler-Maruyama), `logpsi` is identically zero so the entire likelihood signal comes from the canonical message at the root, and one MCMC step is just a few matrix solves rather than 500 SDE Euler steps. Once these pieces are understood, [notebook 07](07_sde_mcmc.ipynb) only changes the transition kernel.